# Optimizer Geometry & Generalization — experiment runner

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.
The core sweep takes roughly 1–2 hours on a free GPU and **saves results as it goes**,
so a dropped connection won't lose finished runs. At the end it zips everything and
downloads it; send that zip back to your chat.

To do a 2-minute test first, set `dataset="synthetic"` and small `epochs`/`seeds` in the
config cell, Run all, confirm it finishes, then switch back to `"FashionMNIST"`.

In [ ]:
# ===================== CONFIG (edit these) =====================
CONFIG = dict(
    dataset="FashionMNIST",          # "FashionMNIST", "CIFAR10", or "synthetic" (quick test)
    batch_sizes=[32, 64, 128, 256, 512, 1024],
    optimizers=["sgd", "adam"],
    seeds=[0, 1, 2],
    epochs=30,
    sgd_lr=0.05, sgd_momentum=0.9,
    adam_lr=1e-3,
    weight_decay=0.0,
    geom_subset=1024,                # fixed # of training examples used for all geometry
    hvp_iters=20,                    # power-iteration steps for top Hessian eigenvalue
    sharp_dirs=10, sharp_rho=0.01,   # random-direction sharpness
    interp_points=21,                # points along the loss-interpolation line
    out_dir="results_geometry",
)

In [ ]:
import os, json, time, random, math, copy, csv, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import grad
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = None  # set inside run_all()


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
DATASET_STATS = {
    "FashionMNIST": dict(ch=1, mean=(0.2860,), std=(0.3530,)),
    "CIFAR10": dict(ch=3, mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616)),
}


def get_data(name):
    if name == "synthetic":
        Xtr = torch.randn(512, 1, 28, 28); ytr = torch.randint(0, 10, (512,))
        Xte = torch.randn(256, 1, 28, 28); yte = torch.randint(0, 10, (256,))
        return TensorDataset(Xtr, ytr), TensorDataset(Xte, yte), 1
    from torchvision import datasets, transforms
    st = DATASET_STATS[name]
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(st["mean"], st["std"])])
    cls = {"FashionMNIST": datasets.FashionMNIST, "CIFAR10": datasets.CIFAR10}[name]
    tr = cls("./data", train=True, download=True, transform=tf)
    te = cls("./data", train=False, download=True, transform=tf)
    return tr, te, st["ch"]


def geom_batch(train_set, n, device):
    """A fixed subset of n training examples, as one (x, y) batch on `device`."""
    idx = list(range(min(n, len(train_set))))
    xs = torch.stack([train_set[i][0] for i in idx])
    ys = torch.tensor([int(train_set[i][1]) for i in idx])
    return xs.to(device), ys.to(device)


class SmallCNN(nn.Module):
    """Two conv blocks + two FC layers; adaptive pool makes it work for 28x28 and 32x32."""
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

In [ ]:
def tparams(model):
    return [p for p in model.parameters() if p.requires_grad]


def _raw_hvp(model, crit, x, y, vec, params):
    """Hessian-vector product of the mean loss on (x, y)."""
    g = grad(crit(model(x), y), params, create_graph=True)
    flat_g = torch.cat([gi.reshape(-1) for gi in g])
    hv = grad((flat_g * vec).sum(), params, retain_graph=False)
    return torch.cat([h.reshape(-1) for h in hv]).detach()


def top_hessian_eigenvalue(model, crit, x, y, iters=20):
    """Largest Hessian eigenvalue via power iteration on HVPs."""
    params = tparams(model)
    v = torch.randn(sum(p.numel() for p in params), device=x.device); v /= v.norm()
    for _ in range(iters):
        Hv = _raw_hvp(model, crit, x, y, v, params)
        nv = Hv.norm()
        if nv == 0:
            break
        v = Hv / nv
    return float((v * _raw_hvp(model, crit, x, y, v, params)).sum().item())


def random_direction_sharpness(model, crit, x, y, rho=0.01, n_dirs=10):
    """Worst loss increase over n_dirs random unit perturbations of norm rho*||w||."""
    params = tparams(model)
    w0 = parameters_to_vector(params).detach().clone()
    wn = w0.norm()
    with torch.no_grad():
        base = crit(model(x), y).item()
    worst = 0.0
    for _ in range(n_dirs):
        d = torch.randn_like(w0); d /= d.norm()
        vector_to_parameters(w0 + rho * wn * d, params)
        with torch.no_grad():
            worst = max(worst, crit(model(x), y).item() - base)
    vector_to_parameters(w0, params)
    return worst

In [ ]:
@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tl = tc = tn = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        o = model(x); l = crit(o, y)
        tl += l.item() * y.size(0); tc += (o.argmax(1) == y).sum().item(); tn += y.size(0)
    return tl / tn, tc / tn


def train_model(opt_name, bs, seed, train_set, test_set, in_ch, cfg):
    set_seed(seed)
    model = SmallCNN(in_ch).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    if opt_name == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg["sgd_lr"],
                              momentum=cfg["sgd_momentum"], weight_decay=cfg["weight_decay"])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=cfg["adam_lr"], weight_decay=cfg["weight_decay"])
    nw = 2 if DEVICE.type == "cuda" else 0
    pm = DEVICE.type == "cuda"
    tr_loader = DataLoader(train_set, batch_size=bs, shuffle=True, num_workers=nw, pin_memory=pm)
    te_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=nw, pin_memory=pm)
    best_acc, best_state, history = -1.0, None, []
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        for x, y in tr_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            crit(model(x), y).backward()
            opt.step()
        trl, tra = evaluate(model, tr_loader, crit)
        tel, tea = evaluate(model, te_loader, crit)
        history.append(dict(epoch=ep, train_loss=trl, train_acc=tra, test_loss=tel, test_acc=tea))
        if tea > best_acc:
            best_acc, best_state = tea, copy.deepcopy(model.state_dict())
    return model, opt, history, best_acc, history[-1]["test_acc"], best_state

In [ ]:
def save_csv(results, out):
    with open(out / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        w.writeheader(); w.writerows(results)


def agg(results, opt, metric):
    bvals = sorted({r["batch_size"] for r in results if r["optimizer"] == opt})
    m = [np.mean([r[metric] for r in results if r["optimizer"] == opt and r["batch_size"] == b]) for b in bvals]
    s = [np.std([r[metric] for r in results if r["optimizer"] == opt and r["batch_size"] == b]) for b in bvals]
    return np.array(bvals), np.array(m), np.array(s)


def make_plots(results, interp, cfg, out):
    figs = out / "figures"
    plt.figure()
    for o in cfg["optimizers"]:
        b, m, s = agg(results, o, "final_test_acc")
        plt.errorbar(b, m, yerr=s, marker="o", capsize=3, label=o.upper())
    plt.xscale("log", base=2); plt.xlabel("batch size"); plt.ylabel("test accuracy")
    plt.title("Generalization vs batch size"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(figs / "acc_vs_batchsize.png", dpi=140, bbox_inches="tight"); plt.show()

    plt.figure()
    for o in cfg["optimizers"]:
        b, m, s = agg(results, o, "top_hessian")
        plt.errorbar(b, m, yerr=s, marker="o", capsize=3, label=o.upper())
    plt.xscale("log", base=2); plt.xlabel("batch size"); plt.ylabel("top Hessian eigenvalue")
    plt.title("Sharpness vs batch size"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(figs / "hessian_vs_batchsize.png", dpi=140, bbox_inches="tight"); plt.show()

    plt.figure()
    for o in cfg["optimizers"]:
        xs = [r["top_hessian"] for r in results if r["optimizer"] == o]
        ys = [r["final_test_acc"] for r in results if r["optimizer"] == o]
        plt.scatter(xs, ys, alpha=.7, label=o.upper())
    plt.xlabel("top Hessian eigenvalue"); plt.ylabel("test accuracy")
    plt.title("Generalization vs sharpness"); plt.legend(); plt.grid(alpha=.3)
    plt.savefig(figs / "acc_vs_hessian.png", dpi=140, bbox_inches="tight"); plt.show()

    if interp is not None:
        a, l = interp
        plt.figure(); plt.plot(a, l, marker="o")
        plt.axvline(0, ls="--", c="gray"); plt.axvline(1, ls="--", c="gray")
        plt.xlabel("alpha  (0 = SGD small-batch, 1 = Adam large-batch)"); plt.ylabel("loss")
        plt.title("Loss interpolation between two minima"); plt.grid(alpha=.3)
        plt.savefig(figs / "interpolation.png", dpi=140, bbox_inches="tight"); plt.show()
    print("Saved figures to", figs)


def run_interpolation(ref, gx, gy, in_ch, cfg):
    if "sgd_small" not in ref or "adam_large" not in ref:
        print("Interpolation reference models missing; skipping."); return None
    model = SmallCNN(in_ch).to(DEVICE); crit = nn.CrossEntropyLoss()
    w1, w2 = ref["sgd_small"], ref["adam_large"]
    alphas = np.linspace(-0.5, 1.5, cfg["interp_points"])
    params = tparams(model); losses = []
    with torch.no_grad():
        for a in alphas:
            vector_to_parameters((1 - a) * w1 + a * w2, params)
            losses.append(crit(model(gx), gy).item())
    return alphas, losses


def main(cfg, out):
    train_set, test_set, in_ch = get_data(cfg["dataset"])
    gx, gy = geom_batch(train_set, cfg["geom_subset"], DEVICE)
    crit = nn.CrossEntropyLoss()
    runs = [(o, bs, s) for o in cfg["optimizers"] for bs in cfg["batch_sizes"] for s in cfg["seeds"]]
    results, ref, t0 = [], {}, time.time()
    for i, (o, bs, s) in enumerate(runs, 1):
        model, opt, hist, best_acc, final_acc, best_state = train_model(
            o, bs, s, train_set, test_set, in_ch, cfg)
        top = top_hessian_eigenvalue(model, crit, gx, gy, cfg["hvp_iters"])
        sharp = random_direction_sharpness(model, crit, gx, gy, cfg["sharp_rho"], cfg["sharp_dirs"])
        row = dict(dataset=cfg["dataset"], optimizer=o, batch_size=bs, seed=s,
                   final_test_acc=final_acc, best_test_acc=best_acc,
                   final_train_loss=hist[-1]["train_loss"], final_test_loss=hist[-1]["test_loss"],
                   top_hessian=top, sharpness=sharp)
        results.append(row)
        # Save full artifacts incl. Adam's optimizer state -> lets us add the
        # preconditioned-curvature (Cohen) analysis later without re-training.
        torch.save(dict(model_final=model.state_dict(), model_best=best_state,
                        optim_state=opt.state_dict(), history=hist, row=row),
                   out / "checkpoints" / f"{cfg['dataset']}_{o}_bs{bs}_seed{s}.pt")
        if o == "sgd" and bs == min(cfg["batch_sizes"]) and s == cfg["seeds"][0]:
            ref["sgd_small"] = parameters_to_vector(tparams(model)).detach().clone()
        if o == "adam" and bs == max(cfg["batch_sizes"]) and s == cfg["seeds"][0]:
            ref["adam_large"] = parameters_to_vector(tparams(model)).detach().clone()
        json.dump(results, open(out / "results.json", "w"), indent=2)
        print(f"[{i:2d}/{len(runs)}] {o:4s} bs={bs:<4d} seed={s} | "
              f"test_acc={final_acc:.4f}  top_H={top:8.3f}  sharp={sharp:.5f} | "
              f"{time.time() - t0:.0f}s elapsed")
    save_csv(results, out)
    return results, ref, gx, gy, in_ch


def run_all(cfg=CONFIG):
    global OUT
    OUT = Path(cfg["out_dir"])
    (OUT / "figures").mkdir(parents=True, exist_ok=True)
    (OUT / "checkpoints").mkdir(parents=True, exist_ok=True)
    print("Device:", DEVICE, "| dataset:", cfg["dataset"])
    results, ref, gx, gy, in_ch = main(cfg, OUT)
    interp = run_interpolation(ref, gx, gy, in_ch, cfg)
    make_plots(results, interp, cfg, OUT)
    zip_path = shutil.make_archive(str(OUT), "zip", str(OUT))
    print("Zipped results ->", zip_path)
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        print("(Not on Colab; grab the zip from the file browser:", zip_path, ")")
    return results

In [ ]:
run_all(CONFIG)